# 03. Time-Series Cause Model

이 노트북은 최근 14일 생활습관과 피부 점수 흐름을 보고
앞으로 악화될지, 어떤 원인이 컸는지 추정하는 모델을 학습합니다.

비유하면:

- 하루하루 기록은 탐정 수첩
- GRU는 그 수첩을 시간순으로 읽는 탐정
- attention은 "어느 날이 특히 중요했는지" 형광펜으로 표시하는 기능


In [ ]:
!pip install -q -r requirements_colab.txt


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
from pathlib import Path

import torch
from torch.optim import AdamW
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from src.skin_coach.config import DEFAULT_CAUSE_COLUMNS, DEFAULT_RISK_COLUMNS, DEFAULT_TEMPORAL_FEATURES
from src.skin_coach.data import SequenceTargetDataset
from src.skin_coach.models import TemporalCauseModel
from src.skin_coach.utils import load_checkpoint, masked_bce_loss, masked_mse_loss, save_checkpoint, seed_everything

seed_everything(42)
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
device


In [ ]:
PROJECT_ROOT = Path("/content/2026_DNN")
DATA_ROOT = PROJECT_ROOT / "processed"
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/2026_DNN/checkpoints/temporal_model")
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_COLUMNS = DEFAULT_TEMPORAL_FEATURES
RISK_COLUMNS = DEFAULT_RISK_COLUMNS
CAUSE_COLUMNS = DEFAULT_CAUSE_COLUMNS
DELTA_COLUMNS = ["skin_score_delta_14d"]
SEQ_LEN = 14
BATCH_SIZE = 32
EPOCHS = 15
LR = 3e-4
OUTPUT_DIR = DRIVE_OUTPUT_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RESUME_FROM = OUTPUT_DIR / "last.pt"
RESUME_FROM = RESUME_FROM if RESUME_FROM.exists() else None


In [ ]:
train_dataset = SequenceTargetDataset(
    daily_logs_csv=str(DATA_ROOT / "daily_logs.csv"),
    targets_csv=str(DATA_ROOT / "temporal_targets.csv"),
    feature_columns=FEATURE_COLUMNS,
    risk_columns=RISK_COLUMNS,
    cause_columns=CAUSE_COLUMNS,
    delta_columns=DELTA_COLUMNS,
    split="train",
    seq_len=SEQ_LEN,
)
val_dataset = SequenceTargetDataset(
    daily_logs_csv=str(DATA_ROOT / "daily_logs.csv"),
    targets_csv=str(DATA_ROOT / "temporal_targets.csv"),
    feature_columns=FEATURE_COLUMNS,
    risk_columns=RISK_COLUMNS,
    cause_columns=CAUSE_COLUMNS,
    delta_columns=DELTA_COLUMNS,
    split="val",
    seq_len=SEQ_LEN,
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

sample = train_dataset[0]
print("sequence shape:", sample["sequence"].shape)
print("risk target shape:", sample["risk_targets"].shape)
print("cause target shape:", sample["cause_targets"].shape)


In [ ]:
model = TemporalCauseModel(
    input_dim=len(FEATURE_COLUMNS),
    risk_columns=RISK_COLUMNS,
    cause_columns=CAUSE_COLUMNS,
    delta_columns=DELTA_COLUMNS,
    hidden_dim=128,
).to(device)
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

best_val = float("inf")
start_epoch = 1
if RESUME_FROM is not None:
    checkpoint = load_checkpoint(str(RESUME_FROM), model=model, optimizer=optimizer, map_location=device)
    start_epoch = int(checkpoint.get("epoch", 0)) + 1
    best_val = float(checkpoint.get("extra_state", {}).get("best_val_loss", checkpoint.get("metrics", {}).get("val_loss", float("inf"))))
    print("Resume from:", RESUME_FROM, "start_epoch:", start_epoch)


In [ ]:
def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)

    total_loss = 0.0
    total_steps = 0

    for batch in tqdm(loader, leave=False):
        sequence = batch["sequence"].to(device)
        sequence_mask = batch["sequence_mask"].to(device)
        risk_targets = batch["risk_targets"].to(device)
        risk_mask = batch["risk_mask"].to(device)
        cause_targets = batch["cause_targets"].to(device)
        cause_mask = batch["cause_mask"].to(device)
        delta_targets = batch["delta_targets"].to(device)
        delta_mask = batch["delta_mask"].to(device)

        outputs = model(sequence, sequence_mask)

        # 한 모델이 "악화 위험", "원인 기여도", "점수 변화량"을 같이 풀기 때문에
        # 손실도 세 문제의 점수를 더해서 계산합니다.
        loss = torch.tensor(0.0, device=device)
        if "risk_logits" in outputs:
            loss = loss + masked_bce_loss(outputs["risk_logits"], risk_targets, risk_mask)
        if "cause_logits" in outputs:
            loss = loss + masked_bce_loss(outputs["cause_logits"], cause_targets, cause_mask)
        if "delta_pred" in outputs:
            loss = loss + masked_mse_loss(outputs["delta_pred"], delta_targets, delta_mask)

        if training:
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

        total_loss += loss.item()
        total_steps += 1

    return total_loss / max(total_steps, 1)


In [ ]:
for epoch in range(start_epoch, EPOCHS + 1):
    train_loss = run_epoch(model, train_loader, optimizer)
    with torch.no_grad():
        val_loss = run_epoch(model, val_loader, optimizer=None)

    print(f"[Epoch {epoch}] train_loss={train_loss:.4f} val_loss={val_loss:.4f}")

    save_checkpoint(
        path=str(OUTPUT_DIR / "last.pt"),
        model=model,
        optimizer=optimizer,
        epoch=epoch,
        metrics={"train_loss": train_loss, "val_loss": val_loss},
        extra_state={
            "feature_columns": FEATURE_COLUMNS,
            "risk_columns": RISK_COLUMNS,
            "cause_columns": CAUSE_COLUMNS,
            "delta_columns": DELTA_COLUMNS,
            "best_val_loss": best_val,
        },
    )

    if val_loss < best_val:
        best_val = val_loss
        save_checkpoint(
            path=str(OUTPUT_DIR / "best.pt"),
            model=model,
            optimizer=optimizer,
            epoch=epoch,
            metrics={"train_loss": train_loss, "val_loss": val_loss},
            extra_state={
                "feature_columns": FEATURE_COLUMNS,
                "risk_columns": RISK_COLUMNS,
                "cause_columns": CAUSE_COLUMNS,
                "delta_columns": DELTA_COLUMNS,
                "best_val_loss": best_val,
            },
        )
